# NB 7 — Compounding reliability (why per-step accuracy misleads)
**Goal:** *measure* the effect behind Figure 6 of the white paper: steps that are each ~95% reliable compound to a much lower end-to-end reliability — and a benchmark that scores one step in isolation will overstate readiness.

This notebook is **pure simulation** — no model or API key needed. It's about measurement, not generation.

In [1]:
import random, statistics
random.seed(0)

STEPS = ["Intake", "Retrieve", "Interpret", "Recommend", "Act / document"]
P = 0.95                     # per-step reliability (each step is quite good on its own)
N = 20000                    # simulated patient episodes

def run_episode(p=P):
    "An episode succeeds only if EVERY step succeeds."
    return all(random.random() < p for _ in STEPS)

end_to_end = statistics.mean(run_episode() for _ in range(N))
print(f"Per-step reliability     : {P:.2f}  (what a single-step benchmark sees)")
print(f"Steps in the workflow    : {len(STEPS)}")
print(f"End-to-end (simulated)   : {end_to_end:.2f}  over {N:,} episodes")
print(f"End-to-end (analytic)    : {P**len(STEPS):.2f}  = {P}^{len(STEPS)}")

Per-step reliability     : 0.95  (what a single-step benchmark sees)
Steps in the workflow    : 5
End-to-end (simulated)   : 0.78  over 20,000 episodes
End-to-end (analytic)    : 0.77  = 0.95^5


### Where the reliability leaks: the cumulative curve
Each step multiplies what survives. This is the curve in Figure 6.

In [2]:
cum = 1.0
print(f"{'after step':<18}{'cumulative reliability':>22}")
for s in STEPS:
    cum *= P
    bar = "#" * round(cum * 40)
    print(f"{s:<18}{cum:>10.2f}   {bar}")
print(f"\nA benchmark scoring only 'Interpret' reports ~{P:.2f}.")
print(f"The patient experiences the whole chain: ~{cum:.2f}.")

after step        cumulative reliability
Intake                  0.95   ######################################
Retrieve                0.90   ####################################
Interpret               0.86   ##################################
Recommend               0.81   #################################
Act / document          0.77   ###############################

A benchmark scoring only 'Interpret' reports ~0.95.
The patient experiences the whole chain: ~0.77.


### Two things your team should feel
1. **Longer workflows leak more** — reliability falls as the chain grows.
2. **A verifier/gate buys back reliability** — catching and fixing a fraction of step errors raises the effective per-step rate, which compounds in your favour.

In [3]:
# 1) end-to-end vs. chain length, at a few per-step rates
print("chain length ->     3       5       8")
for p in [0.99, 0.95, 0.90]:
    row = "   ".join(f"{p**k:5.2f}" for k in (3, 5, 8))
    print(f"  per-step {p:.2f}:   {row}")

# 2) a verifier that catches a fraction c of the errors at each step
def effective_p(p, c):        # c = share of step errors caught & fixed
    return p + (1 - p) * c
print("\n5-step end-to-end, per-step 0.95:")
for c in [0.0, 0.5, 0.8, 0.95]:
    ep = effective_p(0.95, c)
    print(f"  verifier catches {c:>4.0%} of errors -> effective step {ep:.3f} -> end-to-end {ep**5:.2f}")

chain length ->     3       5       8
  per-step 0.99:    0.97    0.95    0.92
  per-step 0.95:    0.86    0.77    0.66
  per-step 0.90:    0.73    0.59    0.43

5-step end-to-end, per-step 0.95:
  verifier catches   0% of errors -> effective step 0.950 -> end-to-end 0.77
  verifier catches  50% of errors -> effective step 0.975 -> end-to-end 0.88
  verifier catches  80% of errors -> effective step 0.990 -> end-to-end 0.95
  verifier catches  95% of errors -> effective step 0.998 -> end-to-end 0.99


### Takeaway
Nothing here is pessimism about any single component — every step was 0.95. The point is structural: **the metric has to match what the patient experiences.** Evaluate the consequential end-state across the whole trajectory, not one step in isolation, and treat verification/gating as a reliability lever, not just a safety one. This is why the paper argues for process- and outcome-level, ambiguity-stratified evaluation rather than a single benchmark score.

*Try:* change `P`, add or remove a step in `STEPS`, or raise the verifier's catch-rate, and watch the end-to-end number move.